# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme

## Pipeline Overview

This notebook implements an automated fact-checking pipeline with two sequential stages:

1. **Stage 1 — Evidence Retrieval:** Hybrid BM25 + CrossEncoder retrieval indexes `evidence.json` and retrieves the top-k most relevant passages for each claim. BM25 first retrieves a broad candidate set (top_k×3); a CrossEncoder re-ranks by joint cross-attention relevance (entailment-aware, not topic-similarity).
2. **Stage 2 — Claim Verification:** A DeBERTa-v3-base NLI model (`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`) classifies the claim against retrieved evidence. Section 2.4 fine-tunes this model on the training set with **class-weighted loss** and **NEI pseudo-evidence sampling** to address label imbalance and the NEI collapse problem.

## Upgrades from Baseline

| Component | Baseline | Upgraded |
|-----------|----------|----------|
| Retrieval | BM25 only, top_k=5 | Hybrid BM25 + CrossEncoder re-rank, top_k=10 |
| BM25 preprocessing | lowercase + punctuation strip | + stopword removal + Porter stemming |
| Re-ranker | None | CrossEncoder (cross-attention, entailment-aware) |
| Classification | Zero-shot 3-class NLI — no DISPUTED label; uniform loss | Fine-tuned 4-class NLI — DISPUTED as learned class; class-weighted cross-entropy loss |
| NEI training signal | Gold evidence for all labels (NEI F1=0) | BM25 pseudo-evidence for NEI claims (P1) |
| Evidence scoring | Per-passage + average | Top-3 concatenated with `[SEP]` — joint multi-evidence input (P3) |
| DISPUTED detection | If-then threshold rule (spec violation) | 4th output class via fine-tuned argmax (P0) — always a learned classifier, never a rule |
| Training | 3 epochs, fixed lr, no early stopping | 5 epochs, early stopping (patience=2) on dev macro-F1; lr=2e-5, weight\_decay=0.1, linear warmup |

## Notes for Marker

- OOP class definitions are in **Section 0** so **Run All** works top-to-bottom without undefined errors.
- The model is loaded in **fp16** for inference — designed for Google Colab free tier (NVIDIA T4, 15 GB VRAM).
- Primary evaluation metric: **Harmonic Mean** of claim classification accuracy and evidence retrieval F-score (as per `eval.py`).
- `DISPUTED` is **always predicted by a learned classifier — no hand-crafted if-then rule** (spec-compliant, P0):
  - Genuine 4th output class via argmax over the fine-tuned 4-way softmax head.
- Multi-evidence concatenation (`top-3 passages [SEP]-joined`) lets DeBERTa reason across conflicting signals simultaneously, which is important for DISPUTED and REFUTES.
- Early stopping saves the best checkpoint by dev macro-F1 and reloads it after training.

## 0. Setup & Class Definitions

Run this section **first**. Mounts Google Drive, installs dependencies, imports libraries, and defines all pipeline classes.

### 0.1 Mount Google Drive & Set Working Directory

In [ ]:
import subprocess, os, sys

# ── BRANCH / VARIANT pairing ──────────────────────────────────────────────
# Jerry/pipeline        → BRANCH="Jerry/pipeline",        VARIANT="pipeline"  (cell 7)
# Jerry/SynMap_pipeline → BRANCH="Jerry/SynMap_pipeline", VARIANT="synmap"    (cell 7)
BRANCH = "Jerry/pipeline"   # ← must match VARIANT in cell 7

REPO_DIR = "/content/comp90042"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH,
         "https://github.com/VitaChien/COMP90042_2026.git", REPO_DIR],
        check=True
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Code source: {REPO_DIR}  branch={BRANCH}")

# ── Drive mount — artifact storage (persists across sessions) ─────────────
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/Final_Assignment"
os.chdir(BASE_DIR)
print(f"Artifact dir: {BASE_DIR}")

### 0.2 Install Dependencies

In [2]:
!pip install bm25s sentence-transformers nltk huggingface_hub spacy -q
!python -m spacy download en_core_web_sm -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 97.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# ── Credentials & repo config ─────────────────────────────────────────────
# Colab Secrets setup: open the key icon (🔑) in the left Colab sidebar and add:
#   HF_TOKEN      — from huggingface.co/settings/tokens  (write access)
#
# If you see "Transport endpoint is not connected": Runtime → Reconnect first.
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN     = userdata.get("HF_TOKEN")
HF_USERNAME  = "ColeH0415"

HF_DEBERTA_REPO = f"{HF_USERNAME}/comp90042-deberta-factcheck"
HF_CE_REPO      = f"{HF_USERNAME}/comp90042-crossencoder-factcheck"
HF_BM25_REPO    = f"{HF_USERNAME}/comp90042-bm25s-index"

# ── Branch identity ───────────────────────────────────────────────────────
# Jerry/pipeline        → "pipeline"
# Jerry/SynMap_pipeline → "synmap"
VARIANT = "pipeline"

# ── Absolute Drive paths ──────────────────────────────────────────────────
# Shared: raw data and dense embeddings (identical across branches)
DENSE_CACHE_PATH = os.path.join(BASE_DIR, "dense_embeddings.npy")
# Branch-specific: all artifacts that touch BM25 get a VARIANT suffix
BM25_CACHE_DIR   = os.path.join(BASE_DIR, f"bm25_index_{VARIANT}")
CE_FINETUNED_DIR = os.path.join(BASE_DIR, f"ce-finetuned-{VARIANT}")
CE_PAIRS_PATH    = os.path.join(BASE_DIR, f"ce_train_pairs_{VARIANT}.pkl")
DEBERTA_BEST_DIR = os.path.join(BASE_DIR, f"deberta-best-{VARIANT}")
DEV_PRED_PATH    = os.path.join(BASE_DIR, f"dev-predictions-{VARIANT}.json")
TEST_PRED_PATH   = os.path.join(BASE_DIR, f"predictions-{VARIANT}.json")

# ── Auth guard + login ────────────────────────────────────────────────────
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found in Colab Secrets. Open the key icon (🔑) in the "
        "left sidebar and add HF_TOKEN with write scope, then re-run this cell."
    )
if HF_USERNAME.startswith("your-hf"):
    raise RuntimeError("Set HF_USERNAME to your real HuggingFace username.")

try:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f"HuggingFace Hub authenticated as {HF_USERNAME} ✓")
    print(f"VARIANT: {VARIANT}  →  artifacts suffixed with '_{VARIANT}'")
    print(f"  BM25_CACHE_DIR:   {BM25_CACHE_DIR}")
    print(f"  DEBERTA_BEST_DIR: {DEBERTA_BEST_DIR}")
except Exception as e:
    print(f"HF login failed: {e}")
    raise

HuggingFace Hub authenticated as ColeH0415 ✓


### 0.3 Imports

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import sys
import json
import re
import torch
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import classification_report, f1_score
import numpy as np

# retriever.py / classifier.py are loaded from REPO_DIR (git-cloned in cell 4).

### 0.4 Class: BM25Retriever

**Upgrades (P1):**
- `_preprocess` now applies stopword removal and Porter stemming in addition to lowercasing and punctuation stripping — improves BM25 term matching quality.
- `top_k` default raised from 5 → 10 for higher recall.

In [5]:
from retriever import (
    BM25Retriever,
    DenseRetriever,
    HybridRetriever,
    load_bm25_index,
    train_cross_encoder,
    normalize_claim,
    STOP_WORDS,
    STEMMER,
)
from classifier import (
    NLIClassifier,
    FactCheckPipeline,
    FactCheckDataset,
    LABEL2ID,
    ID2LABEL,
    LABEL_NAMES,
    eval_macro_f1_dev,
    load_deberta_checkpoint,
    train_deberta,
)
from evaluate_retriever import evaluate_retriever, evaluate_retriever_by_label
print("All modules imported successfully.")

# ── Inline fallback: run this block only if the imports above fail ────────
# (e.g. .py files not yet synced to Drive, or running without retriever.py)
# Uncomment and run this cell to exec the .py files directly:
# for _f in ["retriever.py", "classifier.py"]:
#     if os.path.exists(_f):
#         exec(open(_f).read()); print(f"Inlined: {_f}")


All modules imported successfully.


### 0.4b Class: DenseRetriever

Encodes the entire evidence corpus once with `sentence-transformers/all-MiniLM-L6-v2` (384-dim, L2-normalised) and caches the embeddings to Drive.

At query time, retrieves top-k passages by cosine similarity (dot product of normalised vectors). Complements BM25 by handling **vocabulary mismatch**: finds semantically similar passages even when the claim and evidence use different vocabulary.

Embeddings are pre-computed once (~1–5 min depending on corpus size) and loaded from the Drive cache on subsequent runs — no re-encoding needed.

In [ ]:
# DenseRetriever is imported from retriever.py in the cell above.

### 0.4c Class: HybridRetriever

**P2:** Wraps `BM25Retriever` with a CrossEncoder re-ranker.
- BM25 retrieves a broad candidate set (`bm25_candidates = top_k * 3`).
- `cross-encoder/ms-marco-MiniLM-L6-v2` (22M params) re-ranks by joint cross-attention relevance score — attends to *both* claim and passage simultaneously, providing entailment-aware ranking rather than topic similarity.
- CrossEncoder does **not** pre-encode the corpus; it scores each (claim, passage) pair at query time.
- Final top-k is returned after re-ranking.

**Why CrossEncoder over bi-encoder?** Bi-encoder cosine similarity scores topic overlap, not entailment. CrossEncoder attends jointly to both texts and is trained to score passage relevance for the query — much better aligned with fact-checking needs (Nogueira & Cho, 2019; DeHaven & Scott, 2023).

In [ ]:
# HybridRetriever is imported from retriever.py in the cell above.

### 0.5 Class: NLIClassifier

In [ ]:
# NLIClassifier is imported from classifier.py in the cell above.

### 0.6 Class: FactCheckPipeline

**P0 (spec compliance):** `DISPUTED` is predicted by a learned classifier, never a hand-crafted threshold rule.

The fine-tuned 4-class DeBERTa head predicts `DISPUTED` directly — `label = argmax` over the 4-way softmax.

**P3 (multi-evidence):** Instead of scoring each passage independently and averaging,
the top-3 retrieved passages are concatenated with `[SEP]` and scored as a single input.
This lets DeBERTa reason across conflicting signals simultaneously, which is critical for
DISPUTED (where passages both support and refute) and REFUTES detection.

In [ ]:
# FactCheckPipeline is imported from classifier.py in the cell above.

### 0.7 Class: FactCheckDataset

**P1 (NEI pseudo-evidence sampling):** For `NOT_ENOUGH_INFO` training examples, gold evidence
is replaced with top BM25-retrieved passages. These passages are topically related to the claim
but are not entailment-supporting — which is exactly the signal the model needs to learn NEI.
Without this fix, the model always receives gold evidence even for NEI claims and never learns
when evidence is insufficient, causing NEI F1 = 0.00 (Soleimani et al., 2020).

**P2 (class weights):** `__init__` computes and exposes `class_weights` tensor for use in
weighted cross-entropy loss during fine-tuning.

In [ ]:
# FactCheckDataset is imported from classifier.py in the cell above.

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 1.1 Load Evidence Corpus

In [9]:
with open("evidence.json") as f:
    evidence_dict = json.load(f)

print(f"Loaded {len(evidence_dict):,} evidence passages")
for k in list(evidence_dict.keys())[:3]:
    print(f"  {k}: {evidence_dict[k][:90]}")

Loaded 1,208,827 evidence passages
  evidence-0: John Bennet Lawes, English entrepreneur and agricultural scientist
  evidence-1: Lindberg began his professional career at the age of 16, eventually moving to New York Cit
  evidence-2: ``Boston (Ladies of Cambridge)'' by Vampire Weekend


## 1.2 Load Dataset Splits

In [10]:
def load_json_safe(path):
    if not os.path.exists(path):
        print(f"Warning: {path} not found")
        return {}
    with open(path) as f:
        return json.load(f)

train_data_full = load_json_safe("train-claims.json")
dev_data        = load_json_safe("dev-claims.json")
test_data       = load_json_safe("test-claims-unlabelled.json")

# Use the full training split. Do not downsample: the dataset is small, and
# class imbalance is handled later with class_weights in weighted_loss_fn.
train_data = dict(train_data_full)

print(f"Train: {len(train_data):,} | Dev: {len(dev_data):,} | Test: {len(test_data):,}")

Train: 1,228 | Dev: 154 | Test: 153


In [11]:
# Inspect one example and show label distribution
if train_data:
    first_key = next(iter(train_data))
    print(f"Claim ID: {first_key}")
    print(json.dumps(train_data[first_key], indent=2))

for name, split in [("train", train_data), ("dev", dev_data)]:
    if not split:
        continue
    labels = [v["claim_label"] for v in split.values()]
    print(f"\n{name} distribution ({len(split):,} examples):")
    for lbl, cnt in sorted(Counter(labels).items()):
        print(f"  {lbl:<20} {cnt:>5}  ({cnt/len(labels)*100:.1f}%)")

Claim ID: claim-1937
{
  "claim_text": "Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.",
  "claim_label": "DISPUTED",
  "evidences": [
    "evidence-442946",
    "evidence-1194317",
    "evidence-12171"
  ]
}

train distribution (1,228 examples):
  DISPUTED               124  (10.1%)
  NOT_ENOUGH_INFO        386  (31.4%)
  REFUTES                199  (16.2%)
  SUPPORTS               519  (42.3%)

dev distribution (154 examples):
  DISPUTED                18  (11.7%)
  NOT_ENOUGH_INFO         41  (26.6%)
  REFUTES                 27  (17.5%)
  SUPPORTS                68  (44.2%)


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 2.1 Stage 1 — Build BM25 + Dense + CrossEncoder Retrieval Index

CPU-only. Three retrieval components are built in sequence:

1. **BM25Retriever** — keyword index with stopword removal + Porter stemming (loaded from Drive cache if available).
2. **DenseRetriever** — `sentence-transformers/all-MiniLM-L6-v2` encodes the full corpus into 384-dim normalised embeddings (one-time; cached to Drive).
3. **HybridRetriever** — at query time, BM25 top-50 and Dense top-50 are merged via **Reciprocal Rank Fusion (RRF)**, giving a high-recall candidate set that covers both keyword matches and semantic matches. The CrossEncoder then re-ranks the top-60 RRF candidates to top-10.

**Why RRF?** BM25 misses passages with vocabulary mismatch (synonyms/paraphrases). Dense retrieval covers that gap. RRF safely combines both ranked lists without needing score calibration — a document ranked highly by *both* sources scores highest.

In [12]:
# ── Session control — set before any training ──────────────────────────────
# The lecturer confirmed multi-session Colab runs are acceptable (forum #160).
# Change SESSION to indicate what this Colab session does:
#   "full"       — build retriever + train DeBERTa in one session (default)
#   "retriever"  — Session A: build dense index + optionally train CrossEncoder
#   "classifier" — Session B: load saved retriever checkpoints, train DeBERTa
#   "eval"       — Session C: load all checkpoints, skip all training
SESSION = "retriever"
REUSE_DEBERTA_CHECKPOINT = True   # False → force DeBERTa re-training
REUSE_CE_CHECKPOINT      = True   # False → force CrossEncoder re-training

retriever = load_bm25_index(BM25_CACHE_DIR, evidence_dict, hub_repo=HF_BM25_REPO)


Loading BM25 index from Drive: /content/drive/MyDrive/Final_Assignment/bm25_index


In [13]:
import os
cache = "/content/drive/MyDrive/Final_Assignment/dense_embeddings.npy"
if os.path.exists(cache):
    os.remove(cache)
    print("Stale cache deleted.")

In [14]:
# Build DenseRetriever (encodes corpus once; loads from Drive cache if available)
# DenseRetriever.EMBED_MODEL = "BAAI/bge-large-en-v1.5" (upgraded from all-MiniLM in retriever.py)
dense_retriever = DenseRetriever(evidence_dict)
dense_retriever.build_index(cache_path=DENSE_CACHE_PATH)

# Build HybridRetriever: BM25 + Dense (RRF) + CrossEncoder re-ranking
# HybridRetriever.CE_MODEL = "BAAI/bge-reranker-large" (upgraded from bge-reranker-base in retriever.py)
hybrid_retriever = HybridRetriever(evidence_dict, retriever, dense_retriever=dense_retriever)
hybrid_retriever.build_dense_index()
print("Hybrid retriever ready (BM25 + Dense RRF + CrossEncoder).")


Loading encoder: BAAI/bge-base-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

CrossEncoder loaded on cuda (fp16=True).
Hybrid retriever ready (BM25 + Dense RRF + CrossEncoder).


In [15]:
# Smoke-test retrieval — compare BM25-only, Dense-only, Hybrid BM25+Dense+CE
sample_claim = next(iter(dev_data.values()))["claim_text"]
print(f"Claim: {sample_claim}\n")

print("=== BM25-only (top 5) ===")
for i, p in enumerate(retriever.retrieve(sample_claim, top_k=6), 1):
    print(f"[{i}] bm25_score={p['score']:.3f}  id={p['id']}")
    print(f"     {p['text'][:110]}")

print("\n=== Dense-only (top 5) ===")
for i, p in enumerate(dense_retriever.retrieve(sample_claim, top_k=6), 1):
    print(f"[{i}] dense_score={p['score']:.3f}  id={p['id']}")
    print(f"     {p['text'][:110]}")

print("\n=== Hybrid BM25+Dense+CrossEncoder (top 5) ===")
for i, p in enumerate(hybrid_retriever.retrieve(sample_claim, top_k=6), 1):
    ce = p.get("ce_score", float("nan"))
    print(f"[{i}] ce_score={ce:.3f}  id={p['id']}")
    print(f"     {p['text'][:110]}")

Claim: [South Australia] has the most expensive electricity in the world.

=== BM25-only (top 5) ===


BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

[1] bm25_score=7.154  id=evidence-808896
     The class are the first electric trains to operate in South Australia.
[2] bm25_score=6.559  id=evidence-67732
     [citation needed] South Australia has the highest retail price for electricity in the country.
[3] bm25_score=6.519  id=evidence-572512
     "South Australia has the highest power prices in the world".
[4] bm25_score=6.071  id=evidence-684667
     It is one of the world 's most prestigious and expensive hotels.
[5] bm25_score=6.056  id=evidence-949564
     Carl Albert Unbehaun (1851 -- 5 February 1924) was an electrical engineer in South Australia.
[6] bm25_score=5.927  id=evidence-554677
     Power FM (South Australia), a radio station in South Australia, Australia

=== Dense-only (top 5) ===
[1] dense_score=0.863  id=evidence-67732
     [citation needed] South Australia has the highest retail price for electricity in the country.
[2] dense_score=0.860  id=evidence-572512
     "South Australia has the highest power prices in 

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

[1] ce_score=1.000  id=evidence-572512
     "South Australia has the highest power prices in the world".
[2] ce_score=0.995  id=evidence-67732
     [citation needed] South Australia has the highest retail price for electricity in the country.
[3] ce_score=0.750  id=evidence-240255
     It is found in Australia, where it has been recorded from South Australia.
[4] ce_score=0.750  id=evidence-580844
     It is found in Australia, where it has been recorded from South Australia.
[5] ce_score=0.750  id=evidence-589336
     It is found in Australia, where it has been recorded from South Australia.
[6] ce_score=0.750  id=evidence-786054
     It is found in Australia, where it has been recorded from South Australia.


### 2.1b BM25 Gold Recall Diagnostic

Measures how often the gold evidence passages appear inside BM25's candidate set at
various `top_k` values. This is the **ceiling for the CrossEncoder re-ranker** — if gold
passages are not in the candidate set, no re-ranker can recover them.

**Decision rule:** if `recall@30 < 0.50`, raise the BM25 multiplier in `HybridRetriever.retrieve`
from `top_k * 3` to `top_k * 5` before re-running evidence retrieval.

In [ ]:
# Recall diagnostic — gated so it does not run on every execution.
# Results are stable as long as evidence.json and BM25 parameters are unchanged.
# Set True once after changing BM25 k1/b, top_k multipliers, or the evidence corpus.
RUN_RECALL_DIAGNOSTIC = False

if not RUN_RECALL_DIAGNOSTIC:
    print("Recall diagnostic skipped (RUN_RECALL_DIAGNOSTIC=False).")
    print("Last known: BM25 recall@30=0.376 | set True to re-measure.")
else:
    print("Recall diagnostic (train set, non-NEI claims):")
    for kk in (30, 50, 100):
        # BM25-only recall
        bm25_hits = total = 0
        for item in train_data.values():
            if item["claim_label"] == "NOT_ENOUGH_INFO":
                continue
            gold     = set(item.get("evidences", []))
            hit_ids  = set(r["id"] for r in retriever.retrieve(item["claim_text"], top_k=kk))
            bm25_hits += len(gold & hit_ids)
            total     += len(gold)
        bm25_recall = bm25_hits / total if total else 0.0

        # Hybrid recall (BM25 + Dense via RRF)
        hyb_hits = hyb_total = 0
        for item in train_data.values():
            if item["claim_label"] == "NOT_ENOUGH_INFO":
                continue
            gold    = set(item.get("evidences", []))
            bm25_c  = retriever.retrieve(item["claim_text"], top_k=kk)
            dense_c = dense_retriever.retrieve(item["claim_text"], top_k=kk)
            merged  = HybridRetriever._rrf_merge(bm25_c, dense_c)[:kk]
            hit_ids = set(r["id"] for r in merged)
            hyb_hits  += len(gold & hit_ids)
            hyb_total += len(gold)
        hyb_recall = hyb_hits / hyb_total if hyb_total else 0.0

        flag = " ← raise BM25 multiplier to 5x" if kk == 30 and bm25_recall < 0.50 else ""
        print(f"  @{kk:>3}  BM25={bm25_recall:.3f}  Hybrid={hyb_recall:.3f}{flag}")


In [ ]:
from evaluate_retriever import evaluate_retriever

# ── 1. BM25 alone (Stage 1a ceiling) ─────────────────────────────────────
print("=" * 50)
print("BM25 only")
print("=" * 50)
_bm25_only = type('R', (), {
    'retrieve': lambda self, claim, top_k=10: retriever.retrieve(claim, top_k=top_k)
})()
evaluate_retriever(_bm25_only, dev_data, top_k_list=[10, 20, 50])

# ── 2. Dense alone (Stage 1b ceiling) ────────────────────────────────────
print("=" * 50)
print("Dense only")
print("=" * 50)
_dense_only = type('R', (), {
    'retrieve': lambda self, claim, top_k=10: dense_retriever.retrieve(claim, top_k=top_k)
})()
evaluate_retriever(_dense_only, dev_data, top_k_list=[10, 20, 50])

# ── 3. RRF fusion, no CE (Stage 1+2 ceiling) ─────────────────────────────
print("=" * 50)
print("BM25 + Dense RRF, no CE  ← this is Recall@k before CE sees anything")
print("=" * 50)
_saved_ce = hybrid_retriever.cross_encoder
hybrid_retriever.cross_encoder = None
evaluate_retriever(hybrid_retriever, dev_data, top_k_list=[10, 20, 50])
hybrid_retriever.cross_encoder = _saved_ce

# ── 4. Full hybrid with CE (actual retriever output) ─────────────────────
print("=" * 50)
print("Full hybrid: BM25 + Dense RRF + CE  ← what eval.py measures")
print("=" * 50)
evaluate_retriever(hybrid_retriever, dev_data, top_k_list=[5, 10, 15])


## 2.2 Stage 2 — Initialise Zero-Shot NLI Classifier

Loads `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` in fp16 on the T4 GPU.
This is the **zero-shot baseline**. Section 2.4 fine-tunes it on the training set.

In [ ]:
# device=-1 keeps DeBERTa on CPU until after training — prevents cuBLAS
# workspace allocation that would consume ~11 GB before the training loop.
classifier = NLIClassifier(
    model_name="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=-1
)

In [ ]:
# Smoke-test: check aggregated scores for one pair
scores = classifier.score_single(
    claim="John Bennet Lawes was an English entrepreneur.",
    premise="John Bennet Lawes, English entrepreneur and agricultural scientist"
)
print(f"Fact-check scores: {scores}")
print(f"Predicted label  : {max(scores, key=scores.get)}")

## 2.3 Assemble Zero-Shot Pipeline

`FactCheckPipeline` concatenates the top-3 retrieved passages and scores them jointly (P3).
At this stage the model is still the **zero-shot 3-class NLI** model; label is argmax over
`SUPPORTS / REFUTES / NOT_ENOUGH_INFO`. `DISPUTED` is added as a 4th class after Section 2.4
fine-tuning hot-swaps the classifier.

**P2+P3:** Pipeline uses `hybrid_retriever` (CrossEncoder re-ranked) and multi-evidence
concatenation. Swap `hybrid_retriever` → `retriever` if dense index was skipped.

In [ ]:
# P0: no disputed_threshold — DISPUTED predicted by 4-class argmax after fine-tuning
# P2: hybrid_retriever uses CrossEncoder re-ranking
# P3: FactCheckPipeline concatenates top-3 passages (see class definition)
pipeline = FactCheckPipeline(
    hybrid_retriever,
    classifier,
    top_k=10,
    ev_return_k=5,   # return top-5 IDs to eval.py; NLI still uses top-3 internally
)

# End-to-end sanity check
sample_claim = next(iter(dev_data.values()))["claim_text"]
label, ev_ids = pipeline.predict(sample_claim)
print(f"Claim : {sample_claim}")
print(f"Label : {label}")
print(f"Evid. : {ev_ids}")

## 2.4 Fine-Tune on Training Set

Replaces the 3-class NLI head with a **4-class head** (SUPPORTS / REFUTES / NOT_ENOUGH_INFO / DISPUTED)
and fine-tunes on `train-claims.json`.

- Input: `claim [SEP] concatenated gold evidence` (or BM25 pseudo-evidence for NEI — P1)
- **P0:** DISPUTED is a genuine 4th output class — `argmax` over 4-way softmax, no threshold rule
- **P1:** NEI training examples use BM25-retrieved passages instead of gold evidence, teaching
  the model what "not enough evidence" looks like
- **P2:** Class-weighted cross-entropy prevents collapse to SUPPORTS (44% of training labels)
- **P4 (defensive regularisation):** `lr=2e-5`, `weight_decay=0.1`, linear warmup over 10% of steps — prevents overfitting on the small (~1 700 example) training set
- **P5:** 5 epochs max with early stopping (patience=2) on dev macro-F1; best checkpoint reloaded
- After training the pipeline `classifier` is hot-swapped to use the fine-tuned weights

In [ ]:
import gc

ft_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Collect GPU models to offload before DeBERTa training
_offload = []
if "hybrid_retriever" in dir() and getattr(hybrid_retriever, "cross_encoder", None) is not None:
    _offload.append(hybrid_retriever.cross_encoder.model)
if "dense_retriever" in dir() and getattr(dense_retriever, "model", None) is not None:
    _offload.append(dense_retriever.model)

ft_model, ft_tokenizer = train_deberta(
    train_data, dev_data, evidence_dict,
    bm25_retriever=retriever,
    deberta_best_dir=DEBERTA_BEST_DIR,
    hf_deberta_repo=HF_DEBERTA_REPO,
    reuse_if_found=REUSE_DEBERTA_CHECKPOINT,
    gpu_models_to_offload=_offload,
)


In [ ]:
# Training loop handled by train_deberta() above.

In [ ]:
# Best checkpoint reload handled by train_deberta() above.

In [ ]:
# Hot-swap fine-tuned weights into the pipeline classifier
classifier.tokenizer = ft_tokenizer
classifier.model     = ft_model.half()
classifier.device    = ft_device
classifier.model.eval()

# 4-class pass-through label map (P0: DISPUTED via argmax — no threshold rule)
classifier.label_map = {
    "supports":        "SUPPORTS",
    "refutes":         "REFUTES",
    "not_enough_info": "NOT_ENOUGH_INFO",
    "disputed":        "DISPUTED",
}
print("Pipeline uses fine-tuned 4-class classifier. DISPUTED predicted via argmax (P0).")

# Restore retrieval models to GPU now that DeBERTa training is done.
_ce_device = "cuda" if torch.cuda.is_available() else "cpu"
if "hybrid_retriever" in dir() and getattr(hybrid_retriever, "cross_encoder", None) is not None:
    hybrid_retriever.cross_encoder.model.to(_ce_device)
    print(f"CrossEncoder restored to {_ce_device}.")
if "dense_retriever" in dir() and getattr(dense_retriever, "model", None) is not None:
    dense_retriever.model.to(_ce_device)
    print(f"DenseRetriever restored to {_ce_device}.")

# Quick sanity check
sample_claim = next(iter(dev_data.values()))["claim_text"]
label, ev_ids = pipeline.predict(sample_claim)
print(f"Sample prediction: {label}")

PUSH_LOADED_CHECKPOINT_TO_HUB  = False
RESAVE_LOADED_CHECKPOINT_LOCAL  = False

if REUSE_DEBERTA_CHECKPOINT:
    if PUSH_LOADED_CHECKPOINT_TO_HUB:
        try:
            ft_model.push_to_hub(HF_DEBERTA_REPO, commit_message="Checkpoint from Drive")
            ft_tokenizer.push_to_hub(HF_DEBERTA_REPO)
            print(f"Pushed to: {HF_DEBERTA_REPO}")
        except Exception as e:
            print(f"Hub push FAILED: {e}")
    if RESAVE_LOADED_CHECKPOINT_LOCAL:
        ft_model.save_pretrained(DEBERTA_BEST_DIR)
        ft_tokenizer.save_pretrained(DEBERTA_BEST_DIR)
        print(f"Saved to {DEBERTA_BEST_DIR}")


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 3.1 Evaluate on Dev Set

In [ ]:
from sklearn.metrics import accuracy_score

LABEL_NAMES = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]

if dev_data:
    # ── Oracle mode (classifier ceiling — gold evidence, no retrieval) ─────────
    print(f"Running ORACLE eval on {len(dev_data):,} dev examples ...")
    oracle_preds, y_true_o, y_pred_o = {}, [], []
    for claim_id, item in dev_data.items():
        label, ev_ids = pipeline.predict(
            item["claim_text"],
            gold_evidence_ids=item.get("evidences")
        )
        oracle_preds[claim_id] = {"claim_label": label, "evidences": ev_ids}
        y_pred_o.append(label)
        y_true_o.append(item["claim_label"])
    print("Oracle eval done.\n")

    # ── Pipeline mode (real system — BM25 + CrossEncoder retrieval) ───────────
    print(f"Running PIPELINE eval on {len(dev_data):,} dev examples ...")
    pipeline_preds, y_true_p, y_pred_p = {}, [], []
    for i, (claim_id, item) in enumerate(dev_data.items()):
        label, ev_ids = pipeline.predict(item["claim_text"])
        pipeline_preds[claim_id] = {"claim_label": label, "evidences": ev_ids}
        y_pred_p.append(label)
        y_true_p.append(item["claim_label"])
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(dev_data)} done")
    print("Pipeline eval done.")
else:
    print("No dev data found.")
    oracle_preds = pipeline_preds = {}
    y_true_o = y_pred_o = y_true_p = y_pred_p = []

In [ ]:
if y_true_o and y_pred_o:
    print("=== ORACLE MODE (classifier ceiling — gold evidence) ===")
    print(classification_report(y_true_o, y_pred_o, labels=LABEL_NAMES, zero_division=0))

if y_true_p and y_pred_p:
    print("=== PIPELINE MODE (real system — BM25 + CrossEncoder retrieval) ===")
    print(classification_report(y_true_p, y_pred_p, labels=LABEL_NAMES, zero_division=0))

if y_true_o and y_true_p:
    a_oracle   = accuracy_score(y_true_o, y_pred_o)
    a_pipeline = accuracy_score(y_true_p, y_pred_p)
    gap        = a_oracle - a_pipeline

    print("=== GAP ANALYSIS ===")
    print(f"Oracle accuracy  : {a_oracle:.3f}  (classifier ceiling — gold evidence)")
    print(f"Pipeline accuracy: {a_pipeline:.3f}  (real system — retrieved evidence)")
    print(f"Retrieval gap    : {gap:.3f}  (oracle − pipeline)")
    print()
    if gap > 0.10:
        print("→ Gap > 0.10: RETRIEVER is the bottleneck.")
        print("  Manually enable RUN_OPTIONAL_EXPERIMENTS and run bottom Section 2.5")
        print("  (Priority 2b: supervised CrossEncoder fine-tuning), then re-run evaluation.")
    elif a_oracle < 0.60:
        print("→ Gap ≤ 0.10 but oracle accuracy < 0.60: CLASSIFIER is the bottleneck.")
        print("  Manually enable RUN_OPTIONAL_EXPERIMENTS and run bottom Section 2.6")
        print("  (Priority 4: LoRA fine-tuning), then re-run evaluation.")
    else:
        print("→ Gap ≤ 0.10 and oracle accuracy ≥ 0.60: system is well-balanced.")
        print("  Record scores in CLAUDE.md §3 and proceed to test predictions.")

In [ ]:
# Official eval.py score — harmonic mean of classification accuracy & evidence F-score
import importlib.util
from argparse import Namespace

if pipeline_preds:
    with open(DEV_PRED_PATH, "w") as f:
        json.dump(pipeline_preds, f, indent=2)

    _eval_py_path = os.path.join(BASE_DIR, "eval.py")
    _gt_path      = os.path.join(BASE_DIR, "dev-claims.json")
    spec = importlib.util.spec_from_file_location("eval_module", _eval_py_path)
    eval_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(eval_module)

    args = Namespace(predictions=DEV_PRED_PATH,
                     groundtruth=_gt_path, verbose=False)
    eval_module.main(args)
else:
    print("No pipeline predictions available — run Section 3.1 first.")

In [ ]:
# ── CE comparison tracker ─────────────────────────────────────────────────
# Records F/A/HM each time Section 3.1 runs. Survives multiple re-runs.
# Label is set automatically based on what CE is loaded in the pipeline.
if '_ce_scores' not in dir():
    _ce_scores = {}

if pipeline_preds and y_true_p:
    from sklearn.metrics import accuracy_score
    import re as _re

    # Detect which CE is active
    _ce_path = getattr(hybrid_retriever.cross_encoder, 'model_name_or_path',
                       str(hybrid_retriever.cross_encoder))
    _label = 'fine-tuned CE' if 'ce-finetuned' in str(_ce_path) else 'zero-shot CE'

    # Re-parse F score from last eval.py output (already printed above)
    _a  = accuracy_score(y_true_p, y_pred_p)
    _ev = pipeline_preds  # already written to DEV_PRED_PATH

    # Read F from the saved predictions via eval.py (already ran above)
    # Store what we have directly from Section 3.1 variables
    _ce_scores[_label] = {
        'pipeline_accuracy': round(_a, 4),
        'oracle_accuracy':   round(accuracy_score(y_true_o, y_pred_o), 4),
        'gap':               round(accuracy_score(y_true_o, y_pred_o) - _a, 4),
    }

    print('\n=== CE COMPARISON LOG ===')
    for _lbl, _sc in _ce_scores.items():
        print(f'  {_lbl:<20} pipeline_A={_sc["pipeline_accuracy"]:.4f}  '
              f'oracle_A={_sc["oracle_accuracy"]:.4f}  gap={_sc["gap"]:.4f}')
    print('(Evidence F is printed by eval.py above — note it manually per run)')


## 3.2 Generate Test Predictions

In [ ]:
if test_data:
    print(f"Generating predictions for {len(test_data):,} test examples ...")
    test_predictions = {}

    for i, (claim_id, item) in enumerate(test_data.items()):
        label, ev_ids = pipeline.predict(item["claim_text"])
        test_predictions[claim_id] = {"claim_label": label, "evidences": ev_ids}
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(test_data)} done")

    with open(TEST_PRED_PATH, "w") as f:
        json.dump(test_predictions, f, indent=2)

    print(f"\nSaved {len(test_predictions)} predictions -> {TEST_PRED_PATH}")
    print(f"Label distribution: {dict(Counter(v['claim_label'] for v in test_predictions.values()))}")
else:
    print("No test data found.")

# Optional Experiments (Manual Run Only)

Sections 2.5–2.8 have been moved here so the normal notebook flow reaches evaluation and test prediction first. The cells below are intentionally gated so **Run All** skips the expensive optional training code.

- **Section 2.5:** Supervised CrossEncoder fine-tuning (`RUN_OPTIONAL_EXPERIMENTS = True`)
To run an optional experiment manually, set `RUN_OPTIONAL_EXPERIMENTS = True` in the next cell, then run only the relevant section cells. Keep it `False` for normal top-to-bottom execution and test prediction generation.

In [ ]:
# Manual gate for optional experiments.
# Leave False for normal Run All. Set True only when manually running Section 2.5.
# NOTE: SESSION is set in Section 2.1 (cell 32). Update it there before starting a new session.
RUN_OPTIONAL_EXPERIMENTS = False   # True → run Section 2.5 (supervised CrossEncoder)
print(f"SESSION                  : {SESSION}")
print(f"RUN_OPTIONAL_EXPERIMENTS : {RUN_OPTIONAL_EXPERIMENTS}")


## 2.5 Optional Priority 2b — Supervised CrossEncoder Fine-Tuning

**Run manually only if gap analysis shows `A_oracle - A_pipeline > 0.10`.**

The zero-shot CrossEncoder (`cross-encoder/ms-marco-MiniLM-L6-v2`) ranks passages by general web-search relevance. Fine-tuning it on `(claim, gold_passage)` positive pairs and `(claim, bm25_non-gold)` hard-negative pairs teaches entailment relevance for this fact-checking task.

**Optional pre-check (BM25 recall):** Before training, you can verify BM25 covers enough gold passages:
```python
hits, total = 0, 0
for item in train_data.values():
    if item["claim_label"] == "NOT_ENOUGH_INFO": continue
    gold = set(item.get("evidences", []))
    retrieved = set(r["id"] for r in retriever.retrieve(item["claim_text"], top_k=30))
    hits += len(gold & retrieved); total += len(gold)
print(f"BM25 top-30 gold recall: {hits/total:.3f}")  # raise top_k to 50 if < 0.50
```

**Step 1** (cell below): Build positive/hard-negative pairs and fine-tune the CrossEncoder.  
**Step 2** (next cell): Reload the fine-tuned model into `hybrid_retriever`. Re-run Section 3.1 to measure improvement.

In [ ]:
# P2b — CrossEncoder fine-tuning via train_cross_encoder() from retriever.py
#
# B1 fix: 1:1 positive:negative hard-negative ratio (neg_pool[:len(gold_ids)])
# B2 fix: best checkpoint tracked across all epochs (not overwritten by last-epoch save)
# CACHE NOTE: ce_train_pairs_{VARIANT}.pkl is variant-specific; delete it from Drive to rebuild.
if not RUN_OPTIONAL_EXPERIMENTS:
    print("Skipped P2b Step 1. Set RUN_OPTIONAL_EXPERIMENTS = True to run manually.")
else:
    import gc
    gc.collect(); torch.cuda.empty_cache()

    _ce_offload = []
    if hasattr(classifier, "model"):
        _ce_offload.append(classifier.model)
    if "ft_model" in dir():
        _ce_offload.append(ft_model)

    ce_model = train_cross_encoder(
        train_data, evidence_dict,
        bm25_retriever=retriever,
        ce_finetuned_dir=CE_FINETUNED_DIR,
        hf_ce_repo=HF_CE_REPO,
        base_model="cross-encoder/nli-deberta-v3-base",
        epochs=3,
        lr=1e-6,
        batch_size=16,
        reuse_if_found=REUSE_CE_CHECKPOINT,
        pairs_cache_path=CE_PAIRS_PATH,
        gpu_models_to_offload=_ce_offload,
    )


In [ ]:
from sentence_transformers import CrossEncoder  # needed for CE reload below
# P2b — Step 2: Reload fine-tuned CrossEncoder into hybrid_retriever
if not RUN_OPTIONAL_EXPERIMENTS:
    print("Skipped P2b Step 2. Set RUN_OPTIONAL_EXPERIMENTS = True to run manually.")
else:
    # pipeline.predict() will now use the fine-tuned model for re-ranking.
    # Invalidate oracle lookup cache so it's rebuilt from the updated retriever.
    _ce_device = "cuda" if torch.cuda.is_available() else "cpu"
    hybrid_retriever.cross_encoder = CrossEncoder(CE_FINETUNED_DIR, device=_ce_device)
    pipeline._ev_lookup = None  # force rebuild of oracle cache
    print("hybrid_retriever updated with fine-tuned CrossEncoder.")

    # Restore classifier to GPU — Section 2.5 moved it to CPU to free VRAM
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    classifier.model = classifier.model.half().to(classifier.device)
    classifier.model.eval()
    print(f"Classifier restored to {classifier.device}.")
    print("Re-run Section 3.1 to measure retrieval improvement.")